# ECHOSHIELD AI — Dataset Poisoning Detection & Model Integrity Lab

**Project type:** Defensive AI/ML security • Anomaly detection • Explainable evaluation

ECHOSHIELD AI is a research-style prototype that simulates label-flipping data poisoning, evaluates its impact on a machine-learning model, and tests simple signals that may help flag suspicious training records.

> **Important:** This notebook is an educational baseline, not a production security product. A detector cannot prove that a record is malicious. Real-world use requires domain review, robust validation, privacy safeguards, and testing against multiple attack types.

### What you will build
1. Generate a reproducible classification dataset.
2. Split it into training and untouched test sets.
3. Simulate a controlled label-flipping attack on a subset of training records.
4. Compare a model trained on clean labels with one trained on poisoned labels.
5. Score suspicious records using out-of-fold prediction disagreement and feature-anomaly detection.
6. Evaluate detection quality against the known simulated poison labels.
7. Visualize results and export a forensic report.


## 1. Install and import libraries

If the imports fail in your environment, run the install cell. In Google Colab, you can run it directly.

In [ ]:
# Uncomment if needed:
# %pip install -q numpy pandas matplotlib scikit-learn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("ECHOSHIELD AI environment ready.")


## 2. Generate a baseline dataset

We create a binary classification dataset with informative and redundant features. The data is synthetic, so no real person's information is involved.

In [ ]:
X, y = make_classification(
    n_samples=2500,
    n_features=14,
    n_informative=8,
    n_redundant=3,
    n_repeated=0,
    n_classes=2,
    class_sep=1.1,
    flip_y=0.01,
    random_state=RANDOM_STATE
)

feature_names = [f"feature_{i+1}" for i in range(X.shape[1])]
data = pd.DataFrame(X, columns=feature_names)
data["label"] = y

print("Dataset shape:", data.shape)
display(data.head())
display(data["label"].value_counts().rename("count").to_frame())


## 3. Create train/test split

The test set remains clean and untouched. This is essential: we use it only to measure how training-data corruption affects generalization.

In [ ]:
X_train, X_test, y_train_clean, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Test labels are kept untouched.")


## 4. Simulate a controlled label-flipping attack

The simulation flips a chosen fraction of training labels. The `is_poisoned` flag is retained only for evaluation; a real detector would not have access to this ground truth.

In [ ]:
POISON_RATE = 0.08  # Try 0.03, 0.08, or 0.15
n_poison = max(1, int(len(y_train_clean) * POISON_RATE))

rng = np.random.default_rng(RANDOM_STATE)
poisoned_indices = rng.choice(len(y_train_clean), size=n_poison, replace=False)

y_train_poisoned = y_train_clean.copy()
y_train_poisoned[poisoned_indices] = 1 - y_train_poisoned[poisoned_indices]

is_poisoned = np.zeros(len(y_train_clean), dtype=int)
is_poisoned[poisoned_indices] = 1

print(f"Poisoned records: {is_poisoned.sum()} / {len(is_poisoned)}")
print(f"Actual poison rate: {is_poisoned.mean():.2%}")


## 5. Measure the impact on model performance

Both models use the same algorithm and features. The only difference is whether the training labels were altered.

In [ ]:
baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)
poisoned_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)

baseline_model.fit(X_train, y_train_clean)
poisoned_model.fit(X_train, y_train_poisoned)

pred_baseline = baseline_model.predict(X_test)
pred_poisoned = poisoned_model.predict(X_test)

def metric_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

impact = pd.DataFrame([
    metric_row("Clean training labels", y_test, pred_baseline),
    metric_row("Poisoned training labels", y_test, pred_poisoned)
])
display(impact.style.format({c: "{:.3f}" for c in ["Accuracy", "Precision", "Recall", "F1"]}))


In [ ]:
ax = impact.set_index("Model")[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0
)
ax.set_title("Model performance: clean vs. poisoned training labels")
ax.set_ylabel("Score")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Detection signal A — out-of-fold label disagreement

For each training record, we generate a prediction from a model that did not train on that record (out-of-fold prediction). A mismatch between the observed training label and the out-of-fold prediction can be a warning signal.

**Limitation:** naturally difficult or borderline examples can also be flagged. Disagreement is evidence for review, not proof of poisoning.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_predictions = cross_val_predict(
    make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    ),
    X_train,
    y_train_poisoned,
    cv=cv,
    method="predict"
)

disagreement_score = (oof_predictions != y_train_poisoned).astype(float)
print("Records with OOF disagreement:", int(disagreement_score.sum()))


## 7. Detection signal B — feature-space anomaly score

Isolation Forest identifies records that look unusual in feature space. This may catch some abnormal inputs, but label-flipping attacks can leave feature values unchanged, so this signal may miss them.

In [ ]:
iso = make_pipeline(
    StandardScaler(),
    IsolationForest(
        n_estimators=250,
        contamination=POISON_RATE,
        random_state=RANDOM_STATE
    )
)
iso.fit(X_train)

# decision_function: lower values are more anomalous; invert to make higher = more suspicious
anomaly_raw = -iso.named_steps["isolationforest"].decision_function(
    iso.named_steps["standardscaler"].transform(X_train)
)
anomaly_score = MinMaxScaler().fit_transform(anomaly_raw.reshape(-1, 1)).ravel()

print("Anomaly scores calculated.")


## 8. Combine signals into a review priority score

We combine the two signals with equal weight for this baseline. The score is a prioritization aid, not a calibrated probability of malicious activity.

In [ ]:
# Rank-based normalization keeps the two signals on a comparable 0–1 scale.
disagreement_rank = pd.Series(disagreement_score).rank(pct=True).to_numpy()
anomaly_rank = pd.Series(anomaly_score).rank(pct=True).to_numpy()

combined_score = 0.70 * disagreement_rank + 0.30 * anomaly_rank

results = pd.DataFrame({
    "record_id": np.arange(len(X_train)),
    "observed_label": y_train_poisoned,
    "oof_prediction": oof_predictions,
    "label_disagreement": disagreement_score,
    "anomaly_score": anomaly_score,
    "combined_risk_score": combined_score,
    "simulated_poison_ground_truth": is_poisoned
}).sort_values("combined_risk_score", ascending=False)

display(results.head(10))


## 9. Evaluate detection quality

Because this is a simulation, we know which records were flipped. We can therefore compare the ranking with ground truth. We select the top *k* records, where *k* equals the number of records poisoned in the simulation.

In [ ]:
k = int(is_poisoned.sum())
top_k = results.head(k)
y_true_flags = results["simulated_poison_ground_truth"].to_numpy()
y_score = results["combined_risk_score"].to_numpy()
y_pred_topk = np.zeros(len(results), dtype=int)
y_pred_topk[:k] = 1

detection_metrics = {
    "Precision@K": precision_score(y_true_flags, y_pred_topk, zero_division=0),
    "Recall@K": recall_score(y_true_flags, y_pred_topk, zero_division=0),
    "F1@K": f1_score(y_true_flags, y_pred_topk, zero_division=0),
    "ROC-AUC": roc_auc_score(y_true_flags, y_score)
}

display(pd.DataFrame([detection_metrics]).T.rename(columns={0: "Score"}).style.format("{:.3f}"))
print(f"Poisoned records recovered in top {k}: {int(top_k['simulated_poison_ground_truth'].sum())} / {k}")


In [ ]:
cm = confusion_matrix(y_true_flags, y_pred_topk)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)
ax.set_title("Top-K detection confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0, 1], ["Clean", "Poisoned"])
ax.set_yticks([0, 1], ["Clean", "Poisoned"])
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")
plt.colorbar(im, ax=ax, label="Records")
plt.tight_layout()
plt.show()


## 10. Visualize suspicious records

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(
    results.loc[results.simulated_poison_ground_truth == 0, "combined_risk_score"],
    bins=25, alpha=0.7, label="Clean"
)
axes[0].hist(
    results.loc[results.simulated_poison_ground_truth == 1, "combined_risk_score"],
    bins=25, alpha=0.7, label="Simulated poisoned"
)
axes[0].set_title("Risk-score distribution")
axes[0].set_xlabel("Combined risk score")
axes[0].set_ylabel("Number of records")
axes[0].legend()

plot_data = results.head(40).sort_values("combined_risk_score")
axes[1].barh(plot_data["record_id"].astype(str), plot_data["combined_risk_score"])
axes[1].set_title("Top 40 records by review priority")
axes[1].set_xlabel("Combined risk score")
axes[1].set_ylabel("Training record ID")

plt.tight_layout()
plt.show()


## 11. Inspect the highest-priority records

Review these rows before taking action. In a real workflow, keep original records immutable and send flagged rows to a human reviewer rather than deleting them automatically.

In [ ]:
top_review = results.head(20).copy()
display(top_review)


## 12. Export a forensic report

The CSV includes scores and simulated ground truth for this experiment. Do not expose ground-truth flags in a real detector output; they exist here only to measure the experiment.

In [ ]:
report_path = "echoshield_forensic_report.csv"
results.to_csv(report_path, index=False)

summary = pd.DataFrame([{
    "experiment": "Synthetic label-flipping simulation",
    "random_state": RANDOM_STATE,
    "poison_rate": POISON_RATE,
    "training_records": len(X_train),
    "simulated_poisoned_records": int(is_poisoned.sum()),
    "clean_model_test_accuracy": accuracy_score(y_test, pred_baseline),
    "poisoned_model_test_accuracy": accuracy_score(y_test, pred_poisoned),
    **detection_metrics
}])
summary_path = "echoshield_experiment_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved:", report_path)
print("Saved:", summary_path)
display(summary)


## 13. Experiments to try next

Change one variable at a time and record the results:

- **Poison rate:** test 3%, 8%, 15%, and 20%.
- **Attack type:** compare label flipping with feature perturbation (create a separate, clearly documented experiment).
- **Detector:** compare Logistic Regression, Random Forest, Isolation Forest, and Local Outlier Factor.
- **Scoring weights:** test different weights for disagreement and anomaly scores.
- **Data size:** test smaller and larger datasets.
- **Robustness:** repeat each experiment with multiple random seeds and report mean and standard deviation.

### Responsible-use checklist
- Use synthetic or explicitly authorized datasets.
- Keep the untouched test set separate from training and detector tuning.
- Do not automatically delete flagged records.
- Report false positives as well as successful detections.
- Do not describe a risk score as proof of malicious intent.

### Suggested next build
Turn this notebook into a Streamlit application with dataset upload, attack simulation controls, detector comparison, a review queue, and downloadable reports.
